In [ ]:
#######

#Loading Necessary Packages

#######

import fastmcp as fm
from fastmcp import FastMCP
from pydantic_ai.toolsets.fastmcp import FastMCPToolset
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext, ModelMessage, ModelSettings, UsageLimitExceeded, UsageLimits
from pydantic_ai import ModelSettings
from pydantic_graph import BaseNode, Graph, GraphRunContext, End
from typing import List, Literal
import logfire
from utilities.Types import Settings, ChatState, Interrogator_Output, Prisoner_1_Output, Prisoner_2_Output, Conversational
from utilities.Instructions import get_interrogator_instructions, get_prisoner_instructions, get_Prisoner1_instructions, get_Prisoner2_instructions, get_summary_instructions
import asyncio
from pydantic_ai.messages import ModelMessage, ToolCallPart, ToolReturnPart
from pydantic_evals import Dataset, Case
from pydantic_evals.evaluators import (
    Evaluator,
    EvaluatorContext,
    EvaluationReason,
    EqualsExpected,
    Contains,
    IsInstance,
    MaxDuration,
    LLMJudge,
    HasMatchingSpan,
    Equals
)
from pydantic import BaseModel
from rich.console import Console
from rich.style import Style
console = Console()
info_style = Style(color="green",bold=True)
from dotenv import load_dotenv
import os
logfire.configure()
load_dotenv()
# from worldbuilder import get_interrogator_instructions


#######

#Loading Agents

#######
import Agents.Agents as Agents
Prisoner_1 = Agents.Prisoner_1
Prisoner_2 = Agents.Prisoner_2
Interrogator_ = Agents.Interrogator_
Summarizer = Agents.Summarizer


#Dependencies
deps = Settings()
deps


#########
#Helper Function
#########

def update_interrogator_state(ctx, outcome):
    ctx.state.reasoning.append(outcome.output.reasoning) #track the reasoning
    ctx.state.decisions.append(outcome.output.decision) #track the decision
    ctx.state.questions.append(outcome.output.question) #track the question
    console.print(f"Turn #: {ctx.state.turns}, Decision: {ctx.state.decisions[-1]}, Question: {ctx.state.questions[-1]}, Reason: {ctx.state.reasoning[-1]} ", style=info_style)


######
# Update Tool State Helper
######
def tool_call_helper(reply, ctx):
    """This helper gives all nodes access to tool call outputs so there is no need for them to be repeated"""
    for message in reply.new_messages():
        for part in message.parts:
            if isinstance(part, ToolReturnPart):
                ctx.state.tool_calls.append(part.tool_name) #tracking for eval
                if part.tool_name not in ctx.state.tool_call_outputs: #do not want to store mulitple tool calls
                    ctx.state.tool_call_outputs[part.tool_name] = str(part.content)
                #(f"Tool {part.tool_name}: Output {part.content}")


##############
#Pydantic Graph
#############


###########
#Orchestrator Node (Interrogator)
##########

class Interrogator(BaseNode[ChatState, None, None]):
    '''
    The interrogator class is the orchestrator for the graph. All decisions ulitmately go through here.
    '''
    async def run(self, ctx: GraphRunContext[ChatState]) -> BaseNode[ChatState, None, None]: #have to specify output type, see more here BaseNode.
################ Turn Tracker
        ctx.state.turns += 1  #Everytime the interrogator gets called, we want to add a turn, to limit the number of loops.
        if ctx.state.turns > ctx.state.max_turns:
            return End(data=ctx.state) #end immediately if hitting max turns
################# Dynamic Prompts
        if ctx.state.reasoning: #If there has already been a turn, previous reasioning will have been added to the state, if so, pass in the history to the interrogator.
            #could pass in the whole message history as well instead of just the most recent. pydantic has a messages history option.
            #prompt = "Refer to your message history and continue to make your decision"
            prompt = f"""
            The previous reasonings: {ctx.state.reasoning}
            The previous decisions: {ctx.state.decisions}
            The previous questions: {ctx.state.questions}
            And the previous responses you recieved is {ctx.state.messages[-1]}

            Now, continue your reasoning and make a decision.
            """

        #if there is not a value in previous make your initial decision, we could have this be a user prompt to start the process.
        else:
            prompt = "Make your initial decision"
####################### Running and Tool Checks
        #dynamically update instructions
        #track phase at each point in the graph
        #also want to call in graph state of tool_calls
        outcome = await Interrogator_.run(user_prompt = prompt, deps= deps, output_type = Interrogator_Output, instructions = get_interrogator_instructions(turns = ctx.state.turns, tool_names = ctx.state.tool_calls, decisions= ctx.state.decisions), usage_limits=UsageLimits(total_tokens_limit=8000)) #see types.py for output types
        ctx.state.messages.append(f"Interrogator: {outcome.output}") #track correspondence
        tool_call_helper(reply = outcome, ctx = ctx)
        
##################### Orchestration
        if outcome.output.decision == "ask_prisoner1": #output is the response, decision is the parameter of the outputype class we defined.
            update_interrogator_state(ctx = ctx, outcome = outcome)
            return Prisoner1Node() #returns prisoner 1 node
        elif outcome.output.decision == "ask_prisoner2":
            update_interrogator_state(ctx = ctx, outcome = outcome)
            return Prisoner2Node() #returns prisoner2 node
        elif outcome.output.decision == "prisoner_1 and prisoner_2": #lets both prisoners communicate.
            update_interrogator_state(ctx = ctx, outcome = outcome)
            return Prisoner_Communication_Node() #create a new node for this, cant call two nodes from a single node, must be sequential.
        elif outcome.output.decision == 'end': #if decision to end or we reach max turns, end graph
            return SummarizerNode() #if end, end the game. Final point of the graph always calls end, interrogator should be start and end.
            #print() final decisions


class Prisoner1Node(BaseNode[ChatState, None, None]): #BaseNode takes in State, Deps
    '''
    Prisoner1's decisions and responses, this node is used for 1:1 communications with the interrogator
    '''
#GraphRunContext defines the current state of the Graph
    async def run(self, ctx: GraphRunContext[ChatState]) -> BaseNode[ChatState, None, None]: #have to specify output type, see more here BaseNode.
        #tools etc, can be defined here at run time
        prompt = f"""
        The previous question from the interrogator: {ctx.state.questions[-1]}

        Now, continue your reasoning and make a decision. Use applicable tools to help make your decision.
        """
        #####tool throttling
        if "nash_equilibria" in ctx.state.tool_call_outputs:
            prompt += f"DO NOT RUN nash_equilibria, find the results here: {ctx.state.tool_call_outputs['nash_equilibria']}"
        
        Prisoner_1_reply = await Prisoner_1.run(user_prompt = prompt, deps= deps, output_type= Prisoner_1_Output, instructions = get_Prisoner1_instructions(tool_names = ctx.state.tool_calls), usage_limits=UsageLimits(total_tokens_limit=8000))
        ctx.state.messages.append(f"Prisoner1: {Prisoner_1_reply.output}") #appends new_messages to the state of the graph, so the next node can see them
        tool_call_helper(reply = Prisoner_1_reply, ctx= ctx)

        console.print(Prisoner_1_reply.new_messages, style=info_style)
        return Interrogator() #Responds to the interrogator, then returns to the interrogator to make a decision


class Prisoner2Node(BaseNode[ChatState, None, None]):
    '''
    Prisoner1's decisions and responses, this node is used for 1:1 communications with the interrogator
    '''
    async def run(self, ctx: GraphRunContext[ChatState]) -> BaseNode[ChatState, None, None]:
        prompt = f"""
        The previous question from the interrogator: {ctx.state.questions[-1]}

        Now, continue your reasoning and make a decision.
        """
        if "nash_equilibria" in ctx.state.tool_call_outputs:
            prompt += f"DO NOT RUN nash_equilibria, find the results here: {ctx.state.tool_call_outputs['nash_equilibria']}"
        #it apparently does not like f strings here in the user_prompt
        # Prisoner 2 is prompted with Agents A's output, and it is defined what it needs to do in the system prompt
        Prisoner_2_Reply = await Prisoner_2.run(user_prompt = prompt, deps=deps, output_type = Prisoner_2_Output, instructions = get_Prisoner2_instructions(tool_names = ctx.state.tool_calls), usage_limits=UsageLimits(total_tokens_limit=8000))
        ctx.state.messages.append(f"Prisoner2: {Prisoner_2_Reply.output}") #adding new to the graph messages
        tool_call_helper(reply = Prisoner_2_Reply, ctx= ctx)

        console.print(Prisoner_2_Reply.new_messages, style=info_style)
        #print(ctx.state.new_messages[-1]) #print response from agent P2
        return Interrogator() #Responds to the interrogator, then returns to the interrogator to make a decision





class Prisoner_Communication_Node(BaseNode[ChatState, None, None]):
    #This nodes role is to facilitate communication between Prisoner1 and Prisoner2

    async def run(self, ctx: GraphRunContext[ChatState]) -> BaseNode[ChatState, None, None]:
        #creating a list to store the prisoner communications
        #thought about doing a state but would need to think that through
        Prisoner_1_response = []
        Prisoner_2_response = []
        # The previous question from the interrogator: {ctx.state.questions[-1]}
        prompt = f"""
        Discuss amongst your fellow prisoner before making a decision to confess or not confess.
        When discussing with the other prisoner in order to benefit yourself you may:
            - Decieve
            - Cooroperate
        In order to get a better outcome for youself.
        If you attempt to trick the other prisoner into staying silent so that you can confess list this in you hidden intent, decision, and goal for the game.
        If you attempt to cooroperate for a better outcome for the both of you fill out my_hidden_intent as cooroperate.
        Fill out hidden_goal with how you would like to end the prisoners dilemma for your current strategy.
        Keep your messages short. 3 sentences. Keep in mind that you don't have much time to talk. You will each be able to send two messages before making a decision.
        """
        # if Prisoner_1_response:
        #     prompt = "Prisoner 2's response to your last message {Prisoner_1_response[-1]} is {Prisoner_2_response[-1]}, respond to prisoner_2"
        if "nash_equilibria" in ctx.state.tool_call_outputs:
            prompt += f"DO NOT RUN nash_equilibria, find the results here: {ctx.state.tool_call_outputs['nash_equilibria']}"
    
        counter = 0
        #we will let them communicate twice
        for i in range(3):
            # TODO: Pass entire conversation
            counter += 1
            #Hiding hideen intents from prisoner to prisoner, only receiving message portion of the state.
            if i == 0:
                Prisoner_1_reply = await Prisoner_1.run(user_prompt = f"{prompt} You are prisoner_1 What is your first message to prisoner_2", instructions = get_Prisoner1_instructions(tool_names = ctx.state.tool_calls), deps= deps, output_type=Conversational, usage_limits=UsageLimits(total_tokens_limit=6000))
                Prisoner_2_reply = await Prisoner_2.run(user_prompt = f"{prompt} You are prisoner_2. Respond to prisoner_1's message to you: {Prisoner_1_reply.output.message}", instructions = get_Prisoner2_instructions(tool_names = ctx.state.tool_calls), deps=deps,output_type=Conversational, usage_limits=UsageLimits(total_tokens_limit=6000)) #agent B is prompted with Agents A's output, and it is defined what it needs to do in the system prompt
                Prisoner_1_response.append(f"Prisoner1: {Prisoner_1_reply.output.message}")
                Prisoner_2_response.append(f"Prisoner2: {Prisoner_2_reply.output.message}")
                tool_call_helper(ctx = ctx, reply = Prisoner_1_reply)
                tool_call_helper(ctx = ctx, reply = Prisoner_2_reply)
            elif i == 1:
                Prisoner_1_reply = await Prisoner_1.run(user_prompt = f"{prompt} You are prisoner_1. Respond to prisoner_2's message to you: {Prisoner_2_reply.output.message}", instructions = get_Prisoner1_instructions(tool_names = ctx.state.tool_calls), deps= deps, output_type=Conversational, usage_limits=UsageLimits(total_tokens_limit=6000))
                Prisoner_2_reply = await Prisoner_2.run(user_prompt = f"{prompt} You are prisoner_2. Respond to prisoner_1's message to you: {Prisoner_1_reply.output.message}", instructions = get_Prisoner2_instructions(tool_names = ctx.state.tool_calls), deps=deps,output_type=Conversational, usage_limits=UsageLimits(total_tokens_limit=6000)) #agent B is prompted with Agents A's output, and it is defined what it needs to do in the system prompt
                Prisoner_1_response.append(f"Prisoner1: {Prisoner_1_reply.output.message}")
                Prisoner_2_response.append(f"Prisoner2: {Prisoner_2_reply.output.message}")
                tool_call_helper(ctx = ctx, reply = Prisoner_1_reply)
                tool_call_helper(ctx = ctx, reply = Prisoner_2_reply)
            else: #if last round, output in format expected by interrogator
                Prisoner_1_reply = await Prisoner_1.run(user_prompt = f"{prompt} You are prisoner_1. Respond to prisoner_2's message to you: {Prisoner_2_response[-1]}", instructions = get_Prisoner1_instructions(tool_names = ctx.state.tool_calls), deps= deps, output_type=Prisoner_1_Output, usage_limits=UsageLimits(total_tokens_limit=6000))
                Prisoner_2_reply = await Prisoner_2.run(user_prompt = f"{prompt} You are prisoner_2. Respond to prisoner_1's message to you: {Prisoner_1_reply.output}", instructions = get_Prisoner2_instructions(tool_names = ctx.state.tool_calls), deps=deps,output_type=Prisoner_2_Output, usage_limits=UsageLimits(total_tokens_limit=6000)) #agent B is prompted with Agents A's output, and it is defined what it needs to do in the system prompt
                Prisoner_1_response.append(f"Prisoner1: {Prisoner_1_reply.output}")
                Prisoner_2_response.append(f"Prisoner2: {Prisoner_2_reply.output}")
                tool_call_helper(ctx = ctx, reply = Prisoner_1_reply)
                tool_call_helper(ctx = ctx, reply = Prisoner_2_reply)
            #hidden prisoner to prisoner state tracking.
            #keeps it out of the interogattors inputs
            ctx.state.prisoner_to_prisoner.append(f"Prisoner_1: {Prisoner_1_reply.output}")
            ctx.state.prisoner_to_prisoner.append(f"Prisoner_2: {Prisoner_2_reply.output}")
            #returning in the terminal
            console.print(Prisoner_1_reply.output, style=info_style)
            console.print(Prisoner_2_reply.output, style=info_style)
            if counter >= 3:
                break
        #adding the final decision to context for tracking
        ctx.state.messages.append(f"Prisoner1 in conversation wanted to: {Prisoner_1_response[-1]}, Prisoner2 in conversation wanted to: {Prisoner_2_response[-1]}")
        return Interrogator()

#have an additional LLM collect final decisions potentially
#have an LLM as a judge, analyze behaviors of each agent.


class SummarizerNode(BaseNode[ChatState,None,None]):
    '''
    This node is going to summarize the entirety of the correspondence between prisoners and the interrogator.
    Once the interrogator decides to end his line of questioning , the conversation will come here, and it will be summarized.
    The goal if this node is to outline the conversation, identify key decision, questions, relationships, sentiments and emergent behaviors.
    '''
    async def run(self, ctx:GraphRunContext[ChatState]) -> End:


    #evaluation check on outputs
        summarization = await Summarizer.run(user_prompt = f"Summarize the following conversation {ctx.state.messages}. The prisoner to prisoner communication that is unkown to the interrogator, is {ctx.state.prisoner_to_prisoner}, analyze how that impacted decisions.", deps=deps, usage_limits=UsageLimits(total_tokens_limit=6000))
        return End(summarization.output)


#Graph, is a pydantic object, generating the execution graph.
simple_graph = Graph(nodes=[Interrogator,Prisoner1Node,Prisoner2Node,Prisoner_Communication_Node, SummarizerNode]) #Graph takes in list of classes, representing nodes, to assemble to graph, see more here pg.Graph?

async def main():
    final_state = await simple_graph.run(start_node=Interrogator(),
        state=ChatState(),
        deps=None,
    )
    #console.print(final_state.output, style=info_style)
    return final_state


final_state = await main()
console.print(final_state.output, style=info_style)


#######
#Evaluations
#######


#validating all avenues were questioned
#validating tool calls run as expected
from utilities.evals import create_decision_dataset, create_tools_dataset, evaluation_task_decisions, evaluation_task_tools, create_llm_judge_dataset,evaluation_task_llm_judge, save_results
import pandas as pd
from worldbuilder import STORY

dataset_decisions = create_decision_dataset(final_state)
dataset_tools = create_tools_dataset(final_state)
dataset_llm_judge = create_llm_judge_dataset(final_state)


# Evaluate and print results
validation_D = await dataset_decisions.evaluate(evaluation_task_decisions)
validation_T = await dataset_tools.evaluate(evaluation_task_tools)
validation_llm = await dataset_llm_judge.evaluate(evaluation_task_llm_judge)

#######
#Storing eval results in a json for further analysis.
######
ls = [validation_D,validation_T,validation_llm]
names = ["validation_D","validation_T","validation_llm"]

#uncomment this and edit the file_path to store testing results
'''for i,val in enumerate(ls):
        save_results(val, results_name=names[i], STORY = STORY, file_path= "evals_test_2.json")'''

validation_D.print(include_reasons=True, include_total_duration=True, include_averages=False)
validation_T.print(include_reasons=True, include_total_duration=True, include_averages=False)
validation_llm.print(include_reasons=True, include_total_duration=True, include_averages=False)




Initializing Backstories...
2
Generating Story
{'crime': 'shoplifting from a big box store.',
 'evidence': 'Blury CCTV camera footage shows a vague figure similar to the '
             'prisoners in the area.',
 'evidence_quality': 2,
 'p1_gender': 1,
 'p1_name': 'George',
 'p1_status': 'You are guilty of this crime. But the other prisoner is '
              'innocent.',
 'p2_gender': 1,
 'p2_name': 'Mat',
 'p2_status': 'You are completely innocent. You have no idea what is going on.',
 'relationship': "You feel neutral toward the other prisoner. You've met "
                 'briefly, but nothing stood out.',
 'severity': 1}


Logfire project URL: ]8;id=141100;https://logfire-us.pydantic.dev/gwils1414/agenticaiwithgametheory\https://logfire-us.pydantic.dev/gwils1414/agenticaiwithgametheory]8;;\

15:33:42.692 run graph simple_graph
15:33:42.692   run node Interrogator
15:33:42.695     Interrogator run
15:33:43.595       chat gpt-4o-mini
15:33:45.190       running 1 tool
15:33:45.191         running tool: find_wikipedia_titles
15:33:45.796       chat gpt-4o-mini
15:33:46.529       running 1 tool
15:33:46.529         running tool: get_page_content
15:33:46.726       chat gpt-4o-mini


Turn #: 1, Decision: ask_prisoner1, Question: What do you think your best option is regarding the crime committed? 
Do you feel it would be better to confess or not?, Reason: I researched the concept of the prisoner's dilemma, 
which explains the strategic interaction between rational agents who can choose to cooperate or betray each other 
for self-gain. This theory provides a framework for understanding the situation involving Prisoner_1 and 
Prisoner_2. Based on this, I need to start the questioning phase by directly asking Prisoner_1. 

15:33:50.097   run node Prisoner1Node
15:33:50.100     Prisoner 1 run
15:33:50.104       chat gpt-4o-mini
15:33:51.421       running 1 tool
15:33:51.421         running tool: nash_equilibria
15:33:51.429       chat gpt-4o-mini


...

Output()

15:34:55.257 evaluate evaluation_task_llm_judge
15:34:55.259   case: LLM as a Judge
15:34:55.260     execute evaluation_task_llm_judge
15:34:55.260   case: LLM Judge Don't Confess
15:34:55.261     execute evaluation_task_llm_judge
15:34:55.262   case: LLM Judge Confess/Don't Confess
15:34:55.263     execute evaluation_task_llm_judge
               case: LLM as a Judge
15:34:55.264     evaluator: LLMJudge
               case: LLM Judge Don't Confess
15:34:55.269     evaluator: LLMJudge
               case: LLM Judge Confess/Don't Confess
15:34:55.269     evaluator: LLMJudge


                     Evaluation Summary: evaluation_task_decisions                     
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Case ID              ┃ Assertions                                    ┃    Durations ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ Decision Validation  │ Completed full decision flow: ✔               │  task: 222µs │
│                      │                                               │ total: 1.9ms │
├──────────────────────┼───────────────────────────────────────────────┼──────────────┤
│ Exact Decision Match │ Skipped Prisoner to Prisoner Communication: ✗ │   task: 97µs │
│                      │                                               │ total: 1.3ms │
└──────────────────────┴───────────────────────────────────────────────┴──────────────┘

   Evaluation Summary: evaluation_task_tools   
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Case ID         ┃ Assertions ┃    Durations ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ Tool Validation │ Equals: ✗  │  task: 216µs │
│                 │            │ total: 1.1ms │
└─────────────────┴────────────┴──────────────┘

                                   Evaluation Summary: evaluation_task_llm_judge                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Case ID                        ┃ Scores                         ┃ Assertions                     ┃    Durations ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ LLM as a Judge                 │ LLMJudge_score: 1.00           │ LLMJudge_pass: ✔               │  task: 106µs │
│                                │   Reason: In the final         │   Reason: In the final         │ total: 15.6s │
│                                │ conversation lines Prisoner1:  │ conversation lines Prisoner1:  │              │
│                                │ decision='Confess' and         │ decision='Confess' and         │              │
│                                │ Prisoner2: decision='Confess', │ Prisoner2: decision='Confess', │              │
│                                │ so both decided to confess in  │ so both decided to confess in  │              │
│                                │ their last decisions.          │ their last decisions.          │              │
│                                │                                │                                │              │
│                                │                                │                                │              │
├────────────────────────────────┼────────────────────────────────┼────────────────────────────────┼──────────────┤
│ LLM Judge Don't Confess        │ LLMJudge_score: 0.000          │ LLMJudge_pass: ✗               │  task: 111µs │
│                                │   Reason: Both Prisoner1 and   │   Reason: Both Prisoner1 and   │ total: 14.2s │
│                                │ Prisoner2's last recorded      │ Prisoner2's last recorded      │              │
│                                │ decisions are 'Confess'        │ decisions are 'Confess'        │              │
│                                │ (Prisoner1: 'Confess';         │ (Prisoner1: 'Confess';         │              │
│                                │ Prisoner2: 'Confess'), so they │ Prisoner2: 'Confess'), so they │              │
│                                │ did not both decide 'Don't     │ did not both decide 'Don't     │              │
│                                │ Confess'.                      │ Confess'.                      │              │
│                                │                                │                                │              │
│                                │                                │                                │              │
├────────────────────────────────┼────────────────────────────────┼────────────────────────────────┼──────────────┤
│ LLM Judge Confess/Don't        │ LLMJudge_score: 0.000          │ LLMJudge_pass: ✗               │  task: 333µs │
│ Confess                        │   Reason: In the final         │   Reason: In the final         │ total: 12.7s │
│                                │ conversation both Prisoner1    │ conversation both Prisoner1    │              │
│                                │ and Prisoner2 have             │ and Prisoner2 have             │              │
│                                │ decision='Confess', so it is   │ decision='Confess', so it is   │              │
│                                │ not true that one decided      │ not true that one decided      │              │
│                                │ 'Confess' and the other 'Don't │ 'Confess' and the other 'Don't │              │
│                                │ Confess'.                      │ Confess'.                      │              │
│                                │                                │                                │              │
│                                │                      